# Day 12 — NumPy: Arrays & Indexing
### Python for Data Science · Module 1 · Topic 1.11

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | Why NumPy at all | 15 min |
| 2 | Creating arrays | 25 min |
| 3 | Indexing & slicing — including views | 30 min |
| 4 | Reshaping | 15 min |
| 5 | Mini build: marks analysis without a loop | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **The shift in thinking today.** Until now, to do something to every number you wrote a
> loop. From today you describe the operation and NumPy applies it to the whole array at
> once. The loops do not just get shorter — they disappear.

In [1]:
import numpy as np
print("numpy", np.__version__)

numpy 1.26.4


---
# 1. Why NumPy at all

## 1.1 A list is flexible — and that flexibility is the problem

In [8]:
import time

n = 1_000_000

# The list way
nums = list(range(n))
t = time.time()
doubled = [x * 2 for x in nums]
list_time = time.time() - t

# The NumPy way
arr = np.arange(n)
t = time.time()
doubled = arr * 2
numpy_time = time.time() - t

print(f"list  : {list_time*1000:7.1f} ms")
print(f"numpy : {numpy_time*1000:7.1f} ms")
print(f"numpy is about {list_time/numpy_time:.0f}x faster")

list  :    16.5 ms
numpy :     2.7 ms
numpy is about 6x faster


A Python list checks the type of every element, one at a time, and each number is a separate
object scattered across memory. A NumPy array stores one type in one continuous block, so
the work happens in compiled C.

> **Remember Day 2's warning about nested loops?** Two nested loops over 1,000 items is a
> million iterations. That slide ended with *"this is exactly why NumPy's vectorised
> operations exist"*. This is that moment — the loop has not been made faster, it has been
> moved out of Python entirely.

## 1.2 The ndarray — one type, one block of memory

In [14]:
# A list can hold anything
mixed_list = [1, "two", 3.0, True, None]
print(mixed_list)

# An array holds ONE type
print(np.array([1, 2, 3]).dtype)        # int64
print(np.array([1, 2.4, 3.5]).dtype)      # float64 - everything promoted
print(np.array([1, "2", "sai"]).dtype)      # <U21 - everything became a string!

[1, 'two', 3.0, True, None]
int64
float64
<U21


### ⚠️ The cost of that speed

In [19]:
a = np.array([1, 2, 3])       # an INT array
a[0] = 3.7                # assigning a float...
print(a)                      # ...silently truncated to 3. No warning.

try:
    a.append(4)
except AttributeError as e:
    print("AttributeError:", e)
    print("  -> an array has a FIXED size. Build a list, convert once at the end.")

[3 2 3]
AttributeError: 'numpy.ndarray' object has no attribute 'append'
  -> an array has a FIXED size. Build a list, convert once at the end.


---
# 2. Creating arrays

| Call | Gives | Use it for |
|---|---|---|
| `np.array([1, 2, 3])` | `[1 2 3]` | data you already have in a list |
| `np.arange(0, 10, 2)` | `[0 2 4 6 8]` | a range — stop **excluded**, like `range()` |
| `np.linspace(0, 1, 5)` | `[0. 0.25 0.5 0.75 1.]` | N evenly spaced points — stop **included** |
| `np.zeros((2, 3))` | a 2×3 of `0.0` | a container to fill in later |
| `np.ones((2, 3))` | a 2×3 of `1.0` | same, or a starting multiplier |
| `np.random.rand(3)` | 3 values in `[0, 1)` | test data and simulations |

In [34]:
print("array    :", np.array([1, 2, 3]))
print("arange   :", np.arange(0, 10, 2))
print("linspace :", np.linspace(0, 1, 5))
print()
print("zeros:\n", np.zeros((7, 7)))
print("ones :\n", np.ones((5, 5)))
print("full :\n", np.full((8, 8),9.8))
print("eye  :\n", np.eye(8))

array    : [1 2 3]
arange   : [0 2 4 6 8]
linspace : [0.   0.25 0.5  0.75 1.  ]

zeros:
 [[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]]
ones :
 [[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]
full :
 [[9.8 9.8 9.8 9.8 9.8 9.8 9.8 9.8]
 [9.8 9.8 9.8 9.8 9.8 9.8 9.8 9.8]
 [9.8 9.8 9.8 9.8 9.8 9.8 9.8 9.8]
 [9.8 9.8 9.8 9.8 9.8 9.8 9.8 9.8]
 [9.8 9.8 9.8 9.8 9.8 9.8 9.8 9.8]
 [9.8 9.8 9.8 9.8 9.8 9.8 9.8 9.8]
 [9.8 9.8 9.8 9.8 9.8 9.8 9.8 9.8]
 [9.8 9.8 9.8 9.8 9.8 9.8 9.8 9.8]]
eye  :
 [[1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1.]]


In [49]:
np.random.seed(42)                       # makes the results repeatable
print("rand    :", np.random.rand(3).round(3))       # uniform [0, 1)
print("randint :", np.random.randint(1, 6, 5))     # 5 ints under 100
print("randn   :", np.random.randn(10).round(3))
print("randint :", np.random.randint(1, 6, 5)) 
print("randint :", np.random.randint(1, 6, 5))       # standard normal

rand    : [0.375 0.951 0.732]
randint : [5 5 2 3 3]
randn   : [-2.011 -0.493  0.393 -0.929  0.08  -0.16   0.022 -0.428 -0.532 -0.117]
randint : [4 3 4 4 1]
randint : [3 5 3 5 1]


### ⚠️ Two things that catch everyone once

In [ ]:
# 1. arange EXCLUDES the stop; linspace INCLUDES it
print("arange(0, 1, 0.25)  ->", np.arange(0, 1, 0.25))
print("linspace(0, 1, 5)   ->", np.linspace(0, 1, 5))

np.linspace(0, 1, 5)
np.ones(2, 3)

# 2. The shape is a TUPLE - note the double brackets
print()
print("zeros((2, 3)) works:\n", np.zeros((2, 3)))
try:
    print(np.ones(2, 3))
except TypeError as e:
    print("\nzeros(2, 3) -> TypeError:", str(e)[:60])

arange(0, 1, 0.25)  -> [0.   0.25 0.5  0.75]
linspace(0, 1, 5)   -> [0.   0.25 0.5  0.75 1.  ]

zeros((2, 3)) works:
 [[0. 0. 0.]
 [0. 0. 0.]]

zeros(2, 3) -> TypeError: Cannot interpret '3' as a data type


## 2.1 What every array can tell you about itself

In [54]:
a = np.array([[1, 2, 3],
              [4, 5.0, 6]])

print("a.shape :", a.shape)     # (2, 3) - 2 rows, 3 columns
print("a.ndim  :", a.ndim)      # 2 dimensions
print("a.size  :", a.size)      # 6 elements in total
print("a.dtype :", a.dtype)     # int64

a.shape : (2, 3)
a.ndim  : 2
a.size  : 6
a.dtype : float64


> **`shape` is the one you will use most.** Almost every error you meet in the next six
> weeks will be a shape mismatch. Printing `.shape` is the first thing to do when something
> does not work.

In [59]:
a = np.array([[1, 2, 3], [4, 5, 6]])       # (2, 3)
b = np.array([[1, 2], [3, 4], [5, 6]]) 
c = np.array([[12, 22], [73.9, 46], [65, 99]])     # (3, 2)

print("a.shape:", a.shape, " b.shape:", b.shape)
try:
    print(c + b)
except ValueError as e:
    print("ValueError:", str(e)[:70])

a.shape: (2, 3)  b.shape: (3, 2)
[[ 13.   24. ]
 [ 76.9  50. ]
 [ 70.  105. ]]


In [63]:
# Changing type: astype makes a COPY
a = np.array([1.7, 2.9, 3.1])
print("original :", a)
print("astype   :", a.astype(int))     # truncated, NOT rounded
print("original :", a, " <- unchanged")

# Setting dtype up front is usually cleaner
print("dtype set:", np.array([1, 2, 3], dtype=float))

original : [1.7 2.9 3.1]
astype   : [1 2 3]
original : [1.7 2.9 3.1]  <- unchanged
dtype set: [1. 2. 3.]


In [65]:
a = np.array([1, 2, 3],dtype = float)
a[0] = 7.5
print(a)

[7.5 2.  3. ]


---
# 3. Indexing & slicing

## 3.1 One dimension — exactly like a list

In [ ]:
a = np.array([10, 20, 30, 40, 50])

print("a[0]    :", a[0])
print("a[-1]   :", a[-1])
print("a[1:4]  :", a[1:4])      # stop excluded, as always
print("a[::2]  :", a[::3])
print("a[::-1] :", a[::-3])

print(a)

a[0]    : 10
a[-1]   : 50
a[1:4]  : [20 30 40]
a[::2]  : [10 40]
a[::-1] : [50 20]
[10 40]


Everything from Day 4 carries over: zero-based positions, negative indexes, `start:stop:step`
with the stop excluded, and `[::-1]` to reverse.

## 3.2 ⚠️ A NumPy slice is a VIEW, not a copy

**This is the single most surprising behaviour in NumPy.**

In [77]:
# A LIST slice is a COPY
lst = [1, 2, 3, 4]
part = lst[1:3]
print(part)
part[0] = 99
print("list  :", lst, " <- unchanged")

# An ARRAY slice is a VIEW
arr = np.array([1, 2, 3, 4])
part = arr[1:3]
main = arr[2:4]
main[1]=146
part[0] = 99
print("array :", arr, " <- CHANGED!")

[2, 3]
list  : [1, 2, 3, 4]  <- unchanged
array : [  1  99   3 146]  <- CHANGED!


In [80]:
# You can ask an array whether it owns its data
arr = np.array([1, 2, 3, 4])
view = arr[1:3]
copy = arr[1:3].copy()

copy[0] = 12
print(arr)

print("arr.base  :", arr.base)          # None - it owns its data
print("view.base :", view.base)         # the original array
print("copy.base :", copy.base)         # None - it is independent

copy[0] = 999
print("after changing the copy, arr is:", arr)

[1 2 3 4]
arr.base  : None
view.base : [1 2 3 4]
copy.base : None
after changing the copy, arr is: [1 2 3 4]


**Why NumPy does this:** copying a slice of a 10 GB array would mean copying gigabytes. A
view is just a new window onto the same memory — instant, and free.

> ### The sixth appearance of one idea
> Day 1's `c = a` · Day 3's mutable default argument · Day 3's mutable argument ·
> Day 4's `b = a` · Day 7's shared class attribute · and now a NumPy view.
>
> Every one is the same sentence: **two names, one piece of data.** What is new today is
> that even *slicing* now shares — so writing `.copy()` when you mean a copy matters more
> here than anywhere else in the course.

## 3.3 Two dimensions: `[row, column]`

In [86]:
m = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print("m[1, 1]    :", m[2, 1])        # 5 - preferred
print("m[1][1]    :", m[2][1])        # 5 - works, but slower
print()
print("m[0]       :", m[0])           # first ROW
print("m[:, 0]    :", m[:, 2])        # first COLUMN
print("m[-1, -1]  :", m[-1, -1])
print()
print("m[0:2, 1:] :\n", m[1:, 1:])

m[1, 1]    : 8
m[1][1]    : 8

m[0]       : [1 2 3]
m[:, 0]    : [3 6 9]
m[-1, -1]  : 9

m[0:2, 1:] :
 [[5 6]
 [8 9]]


> **The colon is the part to practise.** `m[:, 0]` reads as *"every row, column 0"* — which
> is how you pull a single column out of a table. In three sessions that becomes
> `df["age"]` in pandas, and it is the same idea underneath.

## 3.4 Boolean masking — the payoff

In [90]:
marks = np.array([88, 35, 71, 22, 64])

condition_to_pass = marks >= 40
print("the mask   :", mask)           # an array of True/False
print("the passes :", marks[mask])    # only where the mask is True
print("in one go  :", marks[marks >= 40])
print("how many   :", condition_to_pass .sum())   # True counts as 1 - Day 1

the mask   : [ True False  True False  True]
the passes : [88 71 64]
in one go  : [88 71 64]
how many   : 3


In [ ]:
# Combining conditions:  &  is and,  |  is or,  ~  is not
mask = (marks >= 40) & (marks < 80)
print(mask)
print("40 to 79    :", marks[(marks >= 40) & (marks < 80)])
print("fails or 90+:", marks[(marks < 40) | (marks >= 90)])
print("not passing :", marks[~(marks >= 40)])

[False False  True False  True]
40 to 79    : [71 64]
fails or 90+: [35 22]
not passing : [35 22]


### ⚠️ Use `&` and `|`, never `and` / `or`

In [ ]:
try:
    marks[(marks >= 40) and (marks < 80)]
except ValueError as e:
    print("ValueError:", str(e)[:75])
    print("  -> and/or want a single True or False.")
    print("     An array of five booleans is neither, so NumPy refuses to guess.")

In [ ]:
# And ALWAYS bracket each condition - & binds tighter than >=
wrong_attempt = None
try:
    wrong_attempt = marks[marks >= 40 & marks < 80]
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:70]}")
    print("  -> Python evaluated  40 & marks  first. Day 1's precedence lesson.")

---
# 4. Reshaping

In [102]:
a = np.arange(9)
print("original :", a, " shape", a.shape)
print()
print("reshape(2, 2):\n", a.reshape(3, 3))
print()
print("reshape(3, 2):\n", a.reshape(3, 3))

# try:
#     a.reshape(4, 2)
# except ValueError as e:
#     print("\nreshape(4, 2) -> ValueError:", str(e)[:50])

original : [0 1 2 3 4 5 6 7 8]  shape (9,)

reshape(2, 2):
 [[0 1 2]
 [3 4 5]
 [6 7 8]]

reshape(3, 2):
 [[0 1 2]
 [3 4 5]
 [6 7 8]]


In [ ]:
# The -1 trick: "you work this one out"
a = np.arange(6)
print("reshape(2, -1) shape:", a.reshape(2, -1).shape)    # (2, 3)
print("reshape(-1, 1) shape:", a.reshape(-1, 1).shape)    # (6, 1) - a column
print("reshape(-1)    shape:", a.reshape(-1).shape)       # (6,)  - flat

# reshape(-1, 1) is the one scikit-learn will demand constantly in Module 3

In [ ]:
m = np.array([[1, 2, 3], [4, 5, 6]])

print("flatten :", m.flatten())    # 1-D, always a COPY
print("ravel   :", m.ravel())      # 1-D, a VIEW when it can be
print("m.T     :\n", m.T)          # transpose - rows become columns
print()
print("flatten owns its data:", m.flatten().base is None)
print("ravel   is a view    :", m.ravel().base is not None)

---
# 5. Putting it together — marks analysis, no loops

In [ ]:
# 6 students x 3 subjects
marks = np.array([[88, 71, 64],
                  [91, 84, 79],
                  [45, 38, 52],
                  [67, 73, 70],
                  [95, 88, 92],
                  [30, 41, 35]])

print("shape        :", marks.shape)

student_avg = marks.mean(axis=1)          # across COLUMNS -> one per student
subject_avg = marks.mean(axis=0)          # down ROWS      -> one per subject

print("per student  :", student_avg.round(1))
print("per subject  :", subject_avg.round(1))

passed = student_avg >= 40
print("pass count   :", passed.sum())
print("top average  :", student_avg.max().round(1))

python_marks = marks[:, 0]                # first column
print("python > 80  :", python_marks[python_marks > 80])

**`axis` is the hardest idea here.** Remember it as:

- `axis=0` — **down** the rows, giving one value per **column**
- `axis=1` — **across** the columns, giving one value per **row**

The axis you name is the one that *disappears*. On Day 2 this whole analysis would have
been four nested loops.

In [ ]:
# Proof that the axis you name is the one that disappears
print("marks.shape          :", marks.shape)            # (6, 3)
print("mean(axis=0).shape   :", marks.mean(axis=0).shape)  # (3,) - the 6 went
print("mean(axis=1).shape   :", marks.mean(axis=1).shape)  # (6,) - the 3 went

---
# 6. Recap — the twelve things to remember

1. `import numpy as np` — always that alias.
2. An array holds **one type**, in one block of memory.
3. That is why it is fast, and why it cannot grow.
4. `np.zeros((2, 3))` — the shape is a **tuple**.
5. `arange` excludes the stop; `linspace` includes it.
6. `shape`, `ndim`, `size`, `dtype` describe any array.
7. A slice is a **view** — changing it changes the original.
8. Use `.copy()` when you actually want a copy.
9. `m[row, column]`; a bare `:` means all of that axis.
10. `m[:, 0]` is a column; `m[0]` is a row.
11. Masks filter without a loop. Use `&` `|` `~`, not `and` `or` `not`.
12. `reshape` keeps the data; `-1` means "work it out".

---

### 📝 Now open **`Day12_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Rewrite Day 2's times table using `np.arange` and `reshape`.
- Take a 5×4 array and print each row's maximum without a loop.
- Prove a slice is a view, then break the link with `.copy()`.

### Next class — Topic 1.12: Broadcasting & vectorised operations
Arithmetic between arrays of different shapes, and why the loop disappears entirely.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*